## Rasteryzing Building heights and mosaiquing rasters
Rasteryzing buildings at 1m , aligned to the extention of the LST raster.

Mosaiquing trees

### Libraries and general paths

In [ ]:
import rasterio
from rasterio.mask import mask
from rasterio.features import rasterize
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import os
import glob
from rasterio.merge import merge
from shapely.geometry import box, mapping

In [ ]:

# Raster reference
lst_cropped_path = '../../../datos-notebooks/0.LST/LST_cropped_celsius.tif' 

#inputs
# Vectorial
buildings = "../../../datos-notebooks/2.global-data-bologna/2.2.objects-height/GBA_bologna.geojson" # vectors of buildings with height column
# Raster 
trees = "../../../datos-notebooks/2.global-data-bologna/2.2.objects-height/gch_bologna/" # folder with vector files of trees with height column

# Rasters - outputs
rasters_base = '../../../datos-notebooks/2.global-data-bologna/2.2.objects-height/rasterized-1m/'
os.makedirs(rasters_base, exist_ok=True)
building_raster_path = os.path.join(rasters_base, 'buildings_height_1m.tif')
trees_raster_path = os.path.join(rasters_base, 'trees_height_1m.tif')

### Rasterizing biuldings

In [ ]:
# Load LST to get extent and CRS
with rasterio.open(lst_cropped_path) as src_lst:
    lst_crs = src_lst.crs
    bounds = src_lst.bounds  # xmin, ymin, xmax, ymax

# Load buildings
gdf_buildings = gpd.read_file(buildings)
gdf_buildings = gdf_buildings.to_crs(lst_crs)  # match LST CRS

# Define raster grid at 0.5 m
res = 0.5  # pixel size
width = int((bounds.right - bounds.left) / res)
height = int((bounds.top - bounds.bottom) / res)
transform = rasterio.transform.from_origin(bounds.left, bounds.top, res, res)

# Rasterize buildings using 'height' column
building_raster = rasterize(
    [(geom, value) for geom, value in zip(gdf_buildings.geometry, gdf_buildings.height)],
    out_shape=(height, width),
    transform=transform,
    fill=np.nan,
    all_touched=True,
    dtype='float32'
)

# Save raster
with rasterio.open(
    building_raster_path,
    'w',
    driver='GTiff',
    height=building_raster.shape[0],
    width=building_raster.shape[1],
    count=1,
    dtype=building_raster.dtype,
    crs=lst_crs,
    transform=transform
) as dst:
    dst.write(building_raster, 1)

In [ ]:
# Print a window for checking the result
with rasterio.open(building_raster_path) as src:
    window = rasterio.windows.Window(col_off=15000, row_off=7000, width=8000, height=8000)
    data = src.read(1, window=window)

plt.figure(figsize=(8,8))
plt.imshow(data, cmap='viridis')
plt.colorbar(label='Height (m)')
plt.show()

### Mosaiquing trees

In [ ]:
# --- Read all tree height rasters ---
tif_files = glob.glob(os.path.join(trees, "*.tif"))
srcs = [rasterio.open(fp) for fp in tif_files]

# --- Mosaic ---
mosaic, transform = merge(srcs)

# --- Read LST extent ---
with rasterio.open(lst_cropped_path) as lst:
    bounds = lst.bounds
    lst_crs = lst.crs

# Geometry from LST extent
geom = [mapping(box(bounds.left, bounds.bottom, bounds.right, bounds.top))]

# --- Clip mosaic to LST extent ---
with rasterio.io.MemoryFile() as memfile:
    with memfile.open(
        driver="GTiff",
        height=mosaic.shape[1],
        width=mosaic.shape[2],
        count=1,
        dtype=mosaic.dtype,
        crs=lst_crs,
        transform=transform,
    ) as tmp:
        tmp.write(mosaic)
        clipped, clipped_transform = mask(tmp, geom, crop=True)

# --- Save output ---
profile = srcs[0].profile
profile.update(
    height=clipped.shape[1],
    width=clipped.shape[2],
    transform=clipped_transform,
    crs=lst_crs
)

with rasterio.open(trees_raster_path, "w", **profile) as dst:
    dst.write(clipped)

# Close sources
for src in srcs:
    src.close()


In [ ]:
with rasterio.open(trees_raster_path) as src:
    window = rasterio.windows.Window(col_off=10000, row_off=10000, width=8000, height=8000)  
    data = src.read(1, window=window)

plt.figure(figsize=(10,10))
plt.imshow(data, cmap='Greens')
plt.colorbar(label='Height (m)')
plt.show()